## 1. Importation des bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Affichage dans le notebook
%matplotlib inline

## 2. Chargement et nettoyage des données

In [2]:
# Chargement du CSV
df = pd.read_csv(r"C:\Users\AKRE\Documents\FORMATION\FORMATION TTA\DI-Bootcamp-TTA\Week2\Day3\daily\Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv", encoding='latin1')

# Aperçu rapide
print("Forme du dataset :", df.shape)
df.head()

Forme du dataset : (4998, 17)


,Date,Time,Location,Operator,Flight #,Route,AC Type,Registration,cn/ln,Aboard,Aboard Passangers,Aboard Crew,Fatalities,Fatalities Passangers,Fatalities Crew,Ground,Summary
0,9/17/1908,17:18,"Fort Myer, Virginia",Military - U.S. Army,NaN,Demonstration,Wright Flyer III,NaN,1,2.0,1.0,1.0,1.0,1.0,0.0,0.0,"During a demonstration flight, a U.S. Army fly..."
1,9/7/1909,NaN,"Juvisy-sur-Orge, France",NaN,NaN,Air show,Wright Byplane,SC1,NaN,1.0,0.0,1.0,1.0,0.0,0.0,0.0,Eugene Lefebvre was the first pilot to ever be...
2,7/12/1912,6:30,"Atlantic City, New Jersey",Military - U.S. Navy,NaN,Test flight,Dirigible,NaN,NaN,5.0,0.0,5.0,5.0,0.0,5.0,0.0,First U.S. dirigible Akron exploded just offsh...
3,8/6/1913,NaN,"Victoria, British Columbia, Canada",Private,NaN,NaN,Curtiss seaplane,NaN,NaN,1.0,0.0,1.0,1.0,0.0,1.0,0.0,The first fatal airplane accident in Canada oc...
4,9/9/1913,18:30,Over the North Sea,Military - German Navy,NaN,NaN,Zeppelin L-1 (airship),NaN,NaN,20.0,NaN,NaN,14.0,NaN,NaN,0.0,The airship flew into a thunderstorm and encou...


In [ ]:
# Informations générales sur les colonnes
df.info()

In [3]:
# Nombre de valeurs manquantes par colonne
df.isnull().sum()

Date                        0
Time                     1512
Location                    4
Operator                   10
Flight #                 3669
Route                     777
AC Type                    15
Registration              274
cn/ln                     668
Aboard                     18
Aboard Passangers         229
Aboard Crew               226
Fatalities                  8
Fatalities Passangers     242
Fatalities Crew           241
Ground                     42
Summary                    64
dtype: int64

### 2.1 Nettoyage et conversion des types

In [ ]:
# Conversion de la colonne Date en format datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Suppression des lignes sans date valide
df = df.dropna(subset=['Date'])

# Extraction de l'année, du mois et de la décennie
df['Year']   = df['Date'].dt.year
df['Month']  = df['Date'].dt.month
df['Decade'] = (df['Year'] // 10 * 10).astype(int)

print("Lignes après nettoyage des dates :", len(df))

In [6]:
# Nombre de valeurs manquantes par colonne
df.isnull().sum()

Date                        0
Time                     1512
Location                    4
Operator                   10
Flight #                 3669
Route                     777
AC Type                    15
Registration              274
cn/ln                     668
Aboard                     18
Aboard Passangers         229
Aboard Crew               226
Fatalities                  8
Fatalities Passangers     242
Fatalities Crew           241
Ground                     42
Summary                    64
Survivors                  18
Survival_Rate              23
dtype: int64

In [5]:
# Conversion des colonnes numériques (certaines sont stockées comme texte)
num_cols = ['Aboard', 'Aboard Passangers', 'Aboard Crew',
            'Fatalities', 'Fatalities Passangers', 'Fatalities Crew', 'Ground']

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Création des colonnes Survivants et Taux de survie
df['Survivors']     = (df['Aboard'] - df['Fatalities']).clip(lower=0)
df['Survival_Rate'] = df['Survivors'] / df['Aboard']
df['Survival_Rate'] = df['Survival_Rate'].clip(0, 1)

df[['Aboard', 'Fatalities', 'Survivors', 'Survival_Rate']].head()

,Aboard,Fatalities,Survivors,Survival_Rate
0,2.0,1.0,1.0,0.5
1,1.0,1.0,0.0,0.0
2,5.0,5.0,0.0,0.0
3,1.0,1.0,0.0,0.0
4,20.0,14.0,6.0,0.3


## 3. Analyse exploratoire (EDA)

### 3.1 Statistiques de base

In [7]:
# Statistiques descriptives des colonnes principales
df[['Aboard', 'Fatalities', 'Survivors', 'Survival_Rate']].describe().round(2)

,Aboard,Fatalities,Survivors,Survival_Rate
count,4980.00,4990.00,4980.00,4975.00
mean,31.20,22.37,8.79,0.18
std,45.53,35.06,30.65,0.31
min,0.00,0.00,0.00,0.00
25%,7.00,4.00,0.00,0.00
50%,16.00,11.00,0.00,0.00
75%,35.00,25.00,3.00,0.25
max,644.00,583.00,516.00,1.00


In [8]:
# Nombre total de crashs, morts et survivants
total_crashes    = len(df)
total_fatalities = df['Fatalities'].sum()
total_survivors  = df['Survivors'].sum()
total_aboard     = df['Aboard'].sum()
overall_fatality_rate = total_fatalities / total_aboard * 100

print(f"Nombre total de crashs    : {total_crashes:,}")
print(f"Total personnes à bord    : {total_aboard:,.0f}")
print(f"Total décès               : {total_fatalities:,.0f}")
print(f"Total survivants          : {total_survivors:,.0f}")
print(f"Taux de mortalité global  : {overall_fatality_rate:.1f}%")

Nombre total de crashs    : 4,998
Total personnes à bord    : 155,356
Total décès               : 111,644
Total survivants          : 43,795
Taux de mortalité global  : 71.9%


### 3.2 Évolution du nombre de crashs par année

In [ ]:
# Nombre de crashs par année
crashes_per_year = df.groupby('Year').size().reset_index(name='Crashes')

plt.figure(figsize=(14, 5))
plt.plot(crashes_per_year['Year'], crashes_per_year['Crashes'], color='steelblue', linewidth=1.5)
plt.fill_between(crashes_per_year['Year'], crashes_per_year['Crashes'], alpha=0.3, color='steelblue')
plt.title('Nombre de crashs aériens par année (1908–2023)', fontsize=14)
plt.xlabel('Année')
plt.ylabel('Nombre de crashs')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 3.3 Évolution des décès par année

In [ ]:
# Décès par année
fatalities_per_year = df.groupby('Year')['Fatalities'].sum().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(fatalities_per_year['Year'], fatalities_per_year['Fatalities'], color='crimson', linewidth=1.5)
plt.fill_between(fatalities_per_year['Year'], fatalities_per_year['Fatalities'], alpha=0.3, color='crimson')
plt.title('Nombre de décès par année (1908–2023)', fontsize=14)
plt.xlabel('Année')
plt.ylabel('Décès')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 3.4 Nombre de crashs par décennie

In [ ]:
# Crashs par décennie
crashes_per_decade = df.groupby('Decade').size().reset_index(name='Crashes')

plt.figure(figsize=(10, 5))
plt.bar(crashes_per_decade['Decade'].astype(str), crashes_per_decade['Crashes'], color='steelblue', edgecolor='white')
plt.title('Nombre de crashs par décennie', fontsize=14)
plt.xlabel('Décennie')
plt.ylabel('Nombre de crashs')
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Analyse statistique

### 4.1 Distribution des décès et taux de survie

In [ ]:
fatalities = df['Fatalities'].dropna()
survival   = df['Survival_Rate'].dropna()

# Statistiques clés
print("=== Décès par crash ===")
print(f"Moyenne   : {fatalities.mean():.2f}")
print(f"Médiane   : {fatalities.median():.2f}")
print(f"Écart-type: {fatalities.std():.2f}")
print(f"Min       : {fatalities.min():.0f}")
print(f"Max       : {fatalities.max():.0f}")

print()
print("=== Taux de survie par crash ===")
print(f"Moyenne   : {survival.mean():.2%}")
print(f"Médiane   : {survival.median():.2%}")
print(f"Écart-type: {survival.std():.2%}")

### 4.2 Histogramme de la distribution des décès

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(fatalities, bins=50, color='crimson', edgecolor='white', alpha=0.8)
plt.axvline(fatalities.mean(),   color='navy',  linestyle='--', linewidth=2, label=f'Moyenne : {fatalities.mean():.0f}')
plt.axvline(fatalities.median(), color='gold',  linestyle='--', linewidth=2, label=f'Médiane : {fatalities.median():.0f}')
plt.title('Distribution du nombre de décès par crash', fontsize=14)
plt.xlabel('Nombre de décès')
plt.ylabel('Fréquence')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 4.3 Test d'hypothèse — Décès : 1940-1970 vs 1980+

In [ ]:
# Groupes : crashs entre 1940-1970 vs crashs à partir de 1980
groupe_ancien = df[(df['Year'] >= 1940) & (df['Year'] <= 1970)]['Fatalities'].dropna()
groupe_recent = df[df['Year'] >= 1980]['Fatalities'].dropna()

print(f"Groupe 1940–1970 → moyenne : {groupe_ancien.mean():.2f}, n = {len(groupe_ancien)}")
print(f"Groupe 1980+     → moyenne : {groupe_recent.mean():.2f}, n = {len(groupe_recent)}")

# Test de Student (Welch) — ne suppose pas l'égalité des variances
t_stat, p_value = stats.ttest_ind(groupe_ancien, groupe_recent, equal_var=False)

print()
print(f"Statistique t : {t_stat:.4f}")
print(f"Valeur p      : {p_value:.4f}")

if p_value < 0.05:
    print("→ Différence statistiquement significative (p < 0.05)")
else:
    print("→ Pas de différence significative (p ≥ 0.05)")

### 4.4 Calcul de la moyenne, médiane et écart-type par décennie

In [ ]:
stats_par_decennie = df.groupby('Decade')['Fatalities'].agg(
    Moyenne='mean',
    Mediane='median',
    Ecart_type='std',
    Nombre_crashs='count'
).round(2)

print(stats_par_decennie)

## 5. Visualisations

### 5.1 Taux de survie moyen par décennie

In [ ]:
survival_par_decennie = df.groupby('Decade')['Survival_Rate'].mean().reset_index()
survival_par_decennie['Survival_Rate'] *= 100  # en pourcentage

plt.figure(figsize=(10, 5))
plt.bar(survival_par_decennie['Decade'].astype(str),
        survival_par_decennie['Survival_Rate'],
        color='seagreen', edgecolor='white')
plt.title('Taux de survie moyen par décennie (%)', fontsize=14)
plt.xlabel('Décennie')
plt.ylabel('Taux de survie (%)')
plt.ylim(0, 50)
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 5.2 Boxplot des décès par décennie

In [ ]:
# Préparer les données par décennie pour le boxplot
decennies = sorted(df['Decade'].dropna().unique())
data_boxplot = [df[df['Decade'] == d]['Fatalities'].dropna().values for d in decennies]

plt.figure(figsize=(14, 6))
plt.boxplot(data_boxplot, labels=[str(int(d)) for d in decennies], patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.7),
            medianprops=dict(color='red', linewidth=2))
plt.title('Distribution des décès par crash selon la décennie', fontsize=14)
plt.xlabel('Décennie')
plt.ylabel('Décès par crash')
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 5.3 Top 10 des opérateurs avec le plus de crashs

In [ ]:
top10_operateurs = df['Operator'].value_counts().head(10)

plt.figure(figsize=(10, 6))
top10_operateurs.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 10 opérateurs par nombre de crashs', fontsize=14)
plt.xlabel('Nombre de crashs')
plt.ylabel('Opérateur')
plt.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 5.4 Heatmap — Crashs par mois et par décennie

In [ ]:
pivot = df.groupby(['Decade', 'Month']).size().unstack(fill_value=0)

plt.figure(figsize=(12, 7))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.3)
plt.title('Nombre de crashs par mois et par décennie', fontsize=14)
plt.xlabel('Mois')
plt.ylabel('Décennie')
plt.tight_layout()
plt.show()

## 6. Résumé et conclusions

### Principaux résultats

1. **Volume de crashs** : Le dataset contient **4 998 crashs** sur 115 ans. Le pic se situe dans les années **1940–1970**, période d'expansion rapide de l'aviation sans réglementation suffisante.

2. **Décès** : La moyenne est de **22 décès par crash**, mais la médiane est de seulement **11** — la distribution est fortement asymétrique. Quelques catastrophes majeures (ex : 1972 avec 2 796 morts) tirent la moyenne vers le haut.

3. **Taux de survie** : Le taux global de mortalité est de **~72%**. Plus de la moitié des crashs n'ont aucun survivant. Ce taux s'améliore nettement après les années 2000.

4. **Test statistique** : Le test de Student entre les périodes 1940–1970 et 1980+ montre une **différence significative** (p = 0.008). Les crashs modernes impliquent des avions plus grands, donc plus de victimes potentielles — mais ils sont bien moins fréquents.

5. **Saisonnalité** : Il n'y a pas de forte saisonnalité globale. La heatmap montre une densité relativement uniforme sur les mois dans les décennies à forte activité.

### Conclusion

L'aviation est **beaucoup plus sûre aujourd'hui** qu'elle ne l'était au XXe siècle. La baisse du nombre de crashs depuis les années 1980 reflète les progrès en matière de réglementation, de formation des pilotes et de technologie. Les statistiques brutes restent impressionnantes à cause de catastrophes isolées, mais le risque par vol a considérablement diminué.
